# 03. Return and Volatility Analysis
**Project:** Stock Market Data Analyzer  
**Author:** Akash Das  
**Phase:** 7 — Return and Volatility Analysis  
**Status:** Validated Implementation

---

### Purpose and Architecture Scope
This notebook implements the foundational performance and risk analytics for the **Stock Market Data Analyzer** project:
1. **Daily Returns**: Percentage changes between consecutive closing prices ($R_t = P_t / P_{t-1} - 1$).
2. **Cumulative Returns**: Compounded growth trajectory over the full observation period ($W_t = \prod (1 + R_i) - 1$).
3. **Volatility Profiling**: Sample standard deviation of daily returns and annualized volatility (scaling by $\sqrt{252}$).
4. **Drawdown Analysis**: Continuous underwater curve and peak-to-trough Maximum Drawdown (MDD) with timestamp attribution.

### Mathematical Foundations
- **Daily Return**:
  $$R_t = \frac{P_t - P_{t-1}}{P_{t-1}} = \frac{P_t}{P_{t-1}} - 1$$
- **Cumulative Compound Growth**:
  $$\text{Wealth}_t = \prod_{i=1}^t (1 + R_i), \quad \text{Cumulative Return}_t = \text{Wealth}_t - 1$$
- **Annualized Volatility**:
  $$\sigma_{\text{daily}} = \sqrt{\frac{1}{N - 1} \sum_{t=1}^N (R_t - \bar{R})^2}, \quad \sigma_{\text{annual}} = \sigma_{\text{daily}} \times \sqrt{252}$$
- **Drawdown and Maximum Drawdown (MDD)**:
  $$\text{HWM}_t = \max_{0 \le s \le t} \text{Wealth}_s, \quad \text{DD}_t = \frac{\text{Wealth}_t}{\text{HWM}_t} - 1, \quad \text{MDD} = \min_{t} \text{DD}_t$$

> **Strict Phase 7 Boundaries**:
> - Performance and historical risk calculations only.
> - **NO** automated trading signals or buy/sell rules.
> - **NO** Plotly interactive visualizations (Phase 8).
> - **NO** Streamlit dashboard UI (Phase 9).
> - **NO** predictive modeling or report generation (Phases 10-12).


## 2. Imports and Configuration
We import standard mathematical and plotting tools alongside the project's analytical packages.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root 'src/' directory is in sys.path
cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Import stock_analyzer modules
from stock_analyzer.data.loader import load_sample_stock_data, load_stock_csv
from stock_analyzer.data.cleaner import clean_stock_data
from stock_analyzer.analysis.returns import (
    ReturnAnalyzer,
    calculate_daily_returns,
    calculate_cumulative_returns,
    add_return_metrics,
)
from stock_analyzer.analysis.risk import (
    RiskAnalyzer,
    calculate_daily_volatility,
    calculate_annualized_volatility,
    calculate_drawdown_series,
    calculate_max_drawdown,
)

# Configuration
pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)
sns.set_theme(style="whitegrid", font="sans-serif")
plt.rcParams["figure.autolayout"] = True

print(f"Project root resolved to: {project_root}")
print("Analytical modules and dependencies imported successfully.")


## 3. Loading Cleaned Stock Data
We load our cleaned baseline dataset from `data/sample/sample_stock.csv` (75 observations) and enforce data validation.


In [ ]:
print("Loading baseline stock dataset...")
df_raw = load_sample_stock_data()
clean_res = clean_stock_data(df_raw)
df = clean_res.data
df.attrs["ticker"] = "SAMPLE_STOCK"
is_synthetic = bool(df.attrs.get("is_synthetic", True))

print(f"Dataset successfully loaded.")
print(f"Ticker:        {df.attrs.get('ticker')}")
print(f"Synthetic:     {is_synthetic}")
print(f"Observations:  {len(df)}")
min_d = str(pd.to_datetime(df['Date']).min())[:10]
max_d = str(pd.to_datetime(df['Date']).max())[:10]
print(f"Date Range:    {min_d} to {max_d}")


## 4. Daily Returns
Daily return measures the percentage price change from the previous closing price.
- **First observation (t=0)** is preserved as `NaN` (no prior price exists).
- Missing returns are **not** filled with zero to avoid distorting risk metrics.


In [ ]:
return_analyzer = ReturnAnalyzer(df, price_column="Close")
daily_returns = return_analyzer.calculate_daily_returns()

print("=== Daily Returns Inspection ===")
print(f"Total observations:    {len(daily_returns)}")
print(f"Initial NaN count:     {daily_returns.isna().sum()} (Expected: 1)")
print(f"Valid return days:     {daily_returns.notna().sum()}")
print(f"Mean Daily Return:     {daily_returns.mean():.4%}")
print(f"Min Daily Return:      {daily_returns.min():.4%}")
print(f"Max Daily Return:      {daily_returns.max():.4%}")

# Display transition head
print("\nFirst 5 observations:")
head_df = pd.DataFrame({"Date": df["Date"].head(5), "Close": df["Close"].head(5), "Daily_Return": daily_returns.head(5)})
print(head_df.to_string(index=False))

# Static Distribution Visualization
fig, (ax_ts, ax_hist) = plt.subplots(nrows=2, ncols=1, figsize=(10, 7), sharex=False)
dates = pd.to_datetime(df["Date"])

ax_ts.plot(dates, daily_returns * 100.0, color="#2563EB", linewidth=1.2, marker="o", markersize=2.5)
ax_ts.axhline(0.0, color="#64748B", linestyle="--", linewidth=1.0)
ax_ts.set_ylabel("Daily Return (%)", fontsize=10)
ax_ts.set_title("Daily Returns Time Series", fontsize=11, fontweight="bold")
ax_ts.grid(True, linestyle=":", alpha=0.6)

sns.histplot(daily_returns.dropna() * 100.0, bins=25, kde=True, ax=ax_hist, color="#3B82F6", edgecolor="white", alpha=0.7)
ax_hist.axvline(daily_returns.mean() * 100.0, color="#DC2626", linestyle="--", label=f"Mean: {daily_returns.mean():.2%}")
ax_hist.set_xlabel("Daily Return (%)", fontsize=10)
ax_hist.set_ylabel("Frequency", fontsize=10)
ax_hist.set_title("Daily Return Distribution", fontsize=11, fontweight="bold")
ax_hist.legend()
ax_hist.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()


## 5. Cumulative Returns
Cumulative compound return represents the percentage capital growth from an initial investment:
$$\text{Cumulative Return}_t = \prod_{i=1}^t (1 + R_i) - 1$$
**Critical Distinction**: Compound cumulative return is derived through multiplicative product, **not** by summing percentage changes.


In [ ]:
cum_returns = return_analyzer.calculate_cumulative_returns()
wealth_index = return_analyzer.calculate_cumulative_growth(initial_wealth=1000.0)

print("=== Cumulative Return Inspection ===")
print(f"Initial value (t=0):   {cum_returns.iloc[0]} (Preserved NaN)")
print(f"Final Cumulative Ret:  {cum_returns.iloc[-1]:.2%}")
print(f"Ending Wealth ($1000): ${wealth_index.iloc[-1]:.2f}")

# Verification: Compare Compound Return vs. Cumulative Sum
cumsum_diff = daily_returns.dropna().cumsum().iloc[-1]
print(f"Simple Cumulative Sum: {cumsum_diff:.2%} (Distorted by arithmetic summing)")
print(f"True Compound Return:  {cum_returns.iloc[-1]:.2%} (Accurate compounding)")

# Plot Cumulative Return Trajectory
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dates, cum_returns * 100.0, color="#059669", linewidth=2.0, label="Cumulative Compound Return (%)")
ax.axhline(0.0, color="#94A3B8", linestyle="--", linewidth=1.0)
ax.set_title(f"{df.attrs.get('ticker')} — Cumulative Performance Trajectory", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Date", fontsize=10)
ax.set_ylabel("Cumulative Return (%)", fontsize=10)
ax.legend(loc="upper left")
ax.grid(True, linestyle=":", alpha=0.6)
plt.show()


## 6. Daily and Annualized Volatility
Volatility quantifies the dispersion of returns around their mean:
- **Daily Volatility**: Sample standard deviation of daily returns ($ddof=1$).
- **Annualized Volatility**: Scaled by the square root of annual trading days (default: $\sqrt{252}$).
$$\sigma_{\text{annual}} = \sigma_{\text{daily}} \times \sqrt{252}$$


In [ ]:
risk_analyzer = RiskAnalyzer(df, price_column="Close")
daily_vol = risk_analyzer.calculate_daily_volatility(ddof=1)
annual_vol_252 = risk_analyzer.calculate_annualized_volatility(annualization_factor=252, ddof=1)
annual_vol_365 = risk_analyzer.calculate_annualized_volatility(annualization_factor=365, ddof=1)

print("=== Volatility Profiling ===")
print(f"Sample Standard Deviation (Daily):      {daily_vol:.4%}")
print(f"Annualized Volatility (252 Trading Days): {annual_vol_252:.2%}")
print(f"Annualized Volatility (365 Calendar Days):{annual_vol_365:.2%}")


## 7. Drawdown Analysis
Drawdown measures the percentage decline from the running historical peak (High-Water Mark):
$$\text{Drawdown}_t = \frac{\text{Wealth}_t}{\text{HWM}_t} - 1$$
Drawdown is continuously non-positive ($\le 0$). A value of $0$ indicates an all-time high.


In [ ]:
drawdown_series = risk_analyzer.calculate_drawdown_series()

# Plot Underwater Curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(dates, drawdown_series * 100.0, color="#DC2626", linewidth=1.5, label="Drawdown (%)")
ax.fill_between(dates, drawdown_series * 100.0, 0, color="#EF4444", alpha=0.3)
ax.axhline(0.0, color="#475569", linestyle="-", linewidth=1.0)
ax.set_title(f"{df.attrs.get('ticker')} — Historical Drawdown (Underwater Plot)", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Date", fontsize=10)
ax.set_ylabel("Drawdown from Peak (%)", fontsize=10)
ax.legend(loc="lower left")
ax.grid(True, linestyle=":", alpha=0.6)
plt.show()


## 8. Maximum Drawdown (MDD) Interpretation
Maximum Drawdown represents the largest peak-to-trough decline before a new high is attained.
It quantifies the worst-case loss an investor would have experienced over the historical window.


In [ ]:
mdd_res = risk_analyzer.calculate_max_drawdown()

print("=== Maximum Drawdown (MDD) Summary ===")
print(f"Maximum Drawdown:      {mdd_res['max_drawdown_pct']:.2f}%")
print(f"Prior Peak Date:       {mdd_res['peak_date']}")
print(f"Trough Date:           {mdd_res['trough_date']}")
print(f"Peak Wealth Index:     {mdd_res['peak_wealth']:.4f}")
print(f"Trough Wealth Index:   {mdd_res['trough_wealth']:.4f}")


## 9. Summary of Performance and Risk
We consolidate all analytical performance and risk metrics into an integrated summary table.


In [ ]:
summary_metrics = pd.DataFrame({
    "Metric": [
        "Total Observations",
        "Total Cumulative Return",
        "Mean Daily Return",
        "Daily Volatility (Std Dev)",
        "Annualized Volatility (252d)",
        "Maximum Drawdown (MDD)",
        "MDD Peak Date",
        "MDD Trough Date"
    ],
    "Value": [
        f"{len(df)} days",
        f"{cum_returns.iloc[-1]:.2%}",
        f"{daily_returns.mean():.4%}",
        f"{daily_vol:.4%}",
        f"{annual_vol_252:.2%}",
        f"{mdd_res['max_drawdown_pct']:.2f}%",
        f"{mdd_res['peak_date']}",
        f"{mdd_res['trough_date']}"
    ]
})

print("=== Executive Performance & Risk Dashboard ===")
print(summary_metrics.to_string(index=False))


## 10. Limitations and Conclusion
### Limitations of Historical Performance Analytics
1. **Normality Assumptions**: Annualization using $\sqrt{252}$ assumes daily returns are independently and identically distributed (i.i.d.). Real financial asset returns exhibit fat tails (kurtosis) and volatility clustering.
2. **Lookback Bias**: Cumulative performance and drawdown depth depend heavily on the chosen start date.
3. **No Forward Predictive Power**: Historical returns and drawdowns summarize past price paths and do not guarantee future return profiles.

### Transition to Phase 8
The verified performance and risk metrics produced in Phase 7 complete our numerical analytical layer. We now proceed to **Phase 8: Data Visualization**, where we implement comprehensive static publication charts (Matplotlib & Seaborn) and interactive financial candlestick charts (Plotly).
